**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 32: 프로젝트 기획 - 도메인 특화 sLLM 파인튜닝

## 🎯 프로젝트 개요 및 목표

이 노트북에서는 **도메인 특화 sLLM 파인튜닝 프로젝트**를 기획합니다.

### 프로젝트 목표
- 특정 도메인에 최적화된 소형 언어모델(sLLM)을 파인튜닝합니다
- 데이터 수집 → 정제 → 학습 → 평가 → 배포의 **전체 파이프라인**을 경험합니다
- 실제 비즈니스 문제를 해결할 수 있는 **실용적인 AI 시스템**을 구축합니다

### 프로젝트 진행 순서

| 단계 | 노트북 | 내용 |
|------|--------|------|
| 1단계 | **32_project_planning** (현재) | 기획 및 계획 수립 |
| 2단계 | 33_project_data_pipeline | 데이터 파이프라인 구축 |
| 3단계 | 34_project_training | 모델 학습 수행 |
| 4단계 | 35_project_evaluation | 성능 평가 및 개선 |
| 5단계 | 36_project_deployment | 배포 및 어플리케이션 |

### 실습 환경 (RTX 4060)
- **GPU**: NVIDIA RTX 4060 (VRAM 7.7GB)
- **사용 모델**: Qwen2.5-1.5B-Instruct (고정)
- **방법론**: LoRA (FP16, 양자화 미사용)
- ⚠️ **제약**: RTX 4060에서는 bitsandbytes NF4 양자화가 호환되지 않으므로 **베이스 모델을 FP16으로 로드**하고 LoRA만 적용합니다. 3B/7B 및 QLoRA는 실습 범위 밖입니다.

---
## 1️⃣ 도메인 선택 가이드

아래 표를 참고하여 프로젝트 도메인을 선택하세요.

| 도메인 | 예시 태스크 | 데이터 소스 | 난이도 | RTX 4060 적합도 |
|--------|------------|------------|--------|----------------|
| 🏥 **의료/건강** | 증상→질환 매핑, 건강 상담 | 의학 Q&A, 논문 요약 | ⭐⭐⭐ | ✅ (1.5B~3B) |
| ⚖️ **법률** | 법률 용어 해석, 판례 요약 | 법률 상담 데이터, 판례 | ⭐⭐⭐ | ✅ (1.5B~3B) |
| 💰 **금융** | 금융 상품 설명, 투자 조언 | 금융 FAQ, 뉴스 요약 | ⭐⭐ | ✅ (1.5B~3B) |
| 📚 **교육** | 학습 도우미, 문제 풀이 | 교과서, Q&A 데이터 | ⭐⭐ | ✅ (1.5B~3B) |
| 🎧 **고객서비스** | FAQ 응답, 불만 처리 | 고객 상담 로그 | ⭐ | ✅ (1.5B~3B) |
| 💻 **코딩** | 코드 생성, 디버깅 | 코드 데이터셋 | ⭐⭐ | ✅ (1.5B~3B) |
| 🍳 **요리/레시피** | 레시피 추천, 재료 대체 | 요리 데이터 | ⭐ | ✅ (1.5B~3B) |
| 🎮 **게임** | NPC 대화, 게임 가이드 | 게임 위키, 대화 로그 | ⭐⭐ | ✅ (1.5B~3B) |

### 도메인 선택 기준
- **데이터 접근성**: 해당 도메인의 데이터를 충분히 확보할 수 있는가?
- **평가 용이성**: 모델 출력의 품질을 객관적으로 평가할 수 있는가?
- **실용적 가치**: 실제로 사용될 수 있는 서비스인가?
- **개인 관심도**: 꾸준히 작업할 동기가 있는가?

In [ ]:
# 도메인 선택 도우미
# 각 도메인에 대한 정보를 확인하고, 적합한 도메인을 선택하세요

domain_info = {
    "의료/건강": {
        "tasks": ["증상 분석", "건강 상담", "의학 용어 설명", "약물 상호작용 안내"],
        "data_sources": ["의학 Q&A 사이트", "건강 상담 데이터", "의학 교과서 요약"],
        "eval_metrics": ["의학적 정확성", "안전성 (유해 조언 방지)", "이해하기 쉬운 설명"],
        "sample_size": "500~2000개 권장"
    },
    "법률": {
        "tasks": ["법률 용어 해석", "판례 요약", "법률 상담 초안", "계약서 검토"],
        "data_sources": ["법률 상담 데이터", "판례 데이터베이스", "법률 FAQ"],
        "eval_metrics": ["법률적 정확성", "조항 인용 정확도", "명확성"],
        "sample_size": "500~2000개 권장"
    },
    "금융": {
        "tasks": ["금융 상품 설명", "투자 정보 요약", "재무제표 분석", "리스크 안내"],
        "data_sources": ["금융 FAQ", "투자 교육 자료", "금융 뉴스"],
        "eval_metrics": ["정보 정확성", "리스크 고지 포함 여부", "명확성"],
        "sample_size": "500~1500개 권장"
    },
    "교육": {
        "tasks": ["개념 설명", "문제 풀이 도우미", "학습 계획 수립", "퀴즈 생성"],
        "data_sources": ["교과서 내용", "교육 Q&A", "강의 자료"],
        "eval_metrics": ["설명의 정확성", "교육적 적절성", "학습자 수준 맞춤"],
        "sample_size": "500~1500개 권장"
    },
    "고객서비스": {
        "tasks": ["FAQ 자동 응답", "불만 처리", "주문 안내", "기술 지원"],
        "data_sources": ["고객 상담 로그", "FAQ 데이터", "제품 매뉴얼"],
        "eval_metrics": ["응답 적절성", "고객 만족도", "문제 해결률"],
        "sample_size": "300~1000개 권장"
    },
    "코딩": {
        "tasks": ["코드 생성", "버그 수정", "코드 설명", "리팩토링 제안"],
        "data_sources": ["코드 Q&A", "GitHub 이슈", "코딩 튜토리얼"],
        "eval_metrics": ["코드 실행 가능성", "정확성", "코드 품질"],
        "sample_size": "500~2000개 권장"
    }
}

# 도메인 정보 출력
for domain, info in domain_info.items():
    print(f"\n{'='*50}")
    print(f"📌 {domain}")
    print(f"{'='*50}")
    print(f"  태스크: {', '.join(info['tasks'])}")
    print(f"  데이터 소스: {', '.join(info['data_sources'])}")
    print(f"  평가 기준: {', '.join(info['eval_metrics'])}")
    print(f"  권장 데이터 수: {info['sample_size']}")

---
## 2️⃣ 문제 정의 템플릿

프로젝트의 핵심은 **명확한 문제 정의**입니다. 아래 템플릿을 채워보세요.

### 문제 정의의 3요소
1. **문제(Problem)**: 무엇을 해결하려고 하는가?
2. **목표(Goal)**: 어떤 결과를 달성하려고 하는가?
3. **평가 기준(Criteria)**: 어떻게 성공을 측정할 것인가?

In [ ]:
# =====================================================
# 📝 문제 정의
# =====================================================

project_definition = {
    "problem": {
        "domain": "AI/ML 기술",
        "target_task": "AI/ML 관련 기술 질문에 정확하게 답변",
        "pain_point": "소형 모델은 LoRA, Attention 등 전문 개념 설명이 부정확함",
        "target_user": "AI/ML 학습자, 개발자",
    },
    "goal": {
        "primary": "AI/ML 기술 질문에 정확하고 상세한 답변 생성",
        "secondary": "한국어 응답 품질 향상",
        "success_criteria": "학습 후 응답이 학습 전보다 구체적이고 정확해짐",
    },
    "evaluation": {
        "auto_metrics": ["응답 길이", "한국어 비율"],
        "llm_judge": False,
        "human_eval": True,
        "eval_dimensions": ["정확성", "구체성", "이해 용이성"],
    }
}

print("✅ 문제 정의 완료")
print(f"   도메인: {project_definition['problem']['domain']}")
print(f"   태스크: {project_definition['problem']['target_task']}")
print(f"   목표: {project_definition['goal']['primary']}")


### 💡 문제 정의 예시

```python
# 예시: 고객서비스 챗봇
project_definition = {
    "problem": {
        "domain": "고객서비스",
        "target_task": "온라인 쇼핑몰 고객 문의 자동 응답",
        "pain_point": "일반 LLM은 회사 정책/상품 정보를 모르고, 환각 답변이 많음",
        "target_user": "쇼핑몰 고객 및 CS 담당자",
    },
    "goal": {
        "primary": "회사 정책 기반 정확한 고객 응답 생성",
        "secondary": "CS 담당자 업무 부하 30% 감소",
        "success_criteria": "LLM-as-a-Judge 정확성 점수 4.0/5.0 이상",
    },
    "evaluation": {
        "auto_metrics": ["ROUGE-L", "BERTScore"],
        "llm_judge": True,
        "human_eval": True,
        "eval_dimensions": ["정확성", "친절도", "정책 준수", "완결성"],
    }
}
```

---
## 3️⃣ 데이터 전략 수립

파인튜닝 프로젝트의 성패는 **데이터 품질**에 달려있습니다.

### 데이터 전략 고려사항

| 항목 | 선택지 | 권장 |
|------|--------|------|
| 수집 방법 | 크롤링 / 공개 데이터셋 / 합성 생성 | 혼합 사용 |
| 데이터 양 | 최소 300 ~ 최대 5000+ | 500~2000 |
| 데이터 형식 | Alpaca / ShareGPT / Custom | Alpaca 권장 |
| 품질 관리 | 수동 검수 / LLM 필터링 / 규칙 기반 | LLM + 수동 |
| 증강 방법 | 패러프레이징 / 역번역 / Distillation | Distillation |

In [ ]:
# =====================================================
# 📝 데이터 전략
# =====================================================

data_strategy = {
    "source": {
        "primary": "../data/samples/alpaca_ko_sample.json",
        "format": "Alpaca (instruction/input/output)",
        "size": "11건",
    },
    "preprocessing": {
        "cleaning": "빈 필드 제거",
        "format_conversion": "Alpaca → ChatML",
        "augmentation": "없음 (기존 데이터 활용)",
    },
    "split": {
        "train": 0.8,
        "val": 0.2,
    }
}

print("✅ 데이터 전략 정의 완료")
print(f"   소스: {data_strategy['source']['primary']}")
print(f"   크기: {data_strategy['source']['size']}")
print(f"   분할: train {data_strategy['split']['train']} / val {data_strategy['split']['val']}")


### 📌 데이터 형식 예시

**Alpaca 형식** (Instruction Tuning에 적합)
```json
{
    "instruction": "다음 증상에 대해 가능한 질환을 알려주세요.",
    "input": "두통, 발열, 인후통이 3일째 지속됩니다.",
    "output": "말씀하신 증상(두통, 발열, 인후통)은 일반적으로 감기나 독감의 초기 증상일 수 있습니다..."
}
```

**ShareGPT 형식** (대화형에 적합)
```json
{
    "conversations": [
        {"from": "human", "value": "두통, 발열, 인후통이 3일째 지속됩니다."},
        {"from": "gpt", "value": "말씀하신 증상으로 보아 감기나 독감이 의심됩니다..."}
    ]
}
```

---
## 4️⃣ 모델 선택 가이드

RTX 4060 (VRAM 7.7GB) 실습 환경에서는 선택지가 제한됩니다.

### RTX 4060 실측 기준 가이드

| 모델 크기 | 방법론 | 상태 | 비고 |
|----------|--------|------|------|
| **1.5B** | LoRA (FP16) | ✅ **본 프로젝트에서 사용** | 베이스 모델 FP16 로드 + LoRA 어댑터 학습 |
| 1.5B | FFT (Full Fine-Tuning) | ⚠️ 빠듯 | 시도 가능하나 불안정 |
| 3B~ | 모든 방법 | ❌ | VRAM 부족 |
| 모든 크기 | QLoRA (NF4 4bit) | ❌ | RTX 4060에서 bitsandbytes NF4 호환 문제로 사용 불가 |

> ⚠️ **중요**: 본 실습 환경에서는 **4bit 양자화(NF4)를 사용하지 않습니다.**
> 베이스 모델을 FP16으로 로드하고 LoRA만 적용하는 구성으로 Part 5 전체를 진행합니다.

### 본 프로젝트 사용 모델

| 모델 | 크기 | 이유 |
|------|------|------|
| **Qwen2.5-1.5B-Instruct** | 1.5B | RTX 4060 VRAM 7.7GB에서 안정적으로 LoRA 학습 가능한 최대 크기 |

In [ ]:
# =====================================================
# 📝 모델 선택
# =====================================================

model_config = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "method": "LoRA",
    "quantization": "none (FP16)",
    "lora_config": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
    },
    "training": {
        "epochs": 5,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "learning_rate": 2e-4,
        "max_length": 1024,
        "fp16": True,
    },
    "reason": "RTX 4060 VRAM 7.7GB에서 NF4 양자화는 호환 문제가 있어, 베이스를 FP16으로 로드하고 LoRA만 적용",
}

print("✅ 모델 선택 완료")
print(f"   모델: {model_config['base_model']}")
print(f"   방법: {model_config['method']} ({model_config['quantization']})")
print(f"   LoRA r={model_config['lora_config']['r']}, alpha={model_config['lora_config']['alpha']}")
print(f"   에포크: {model_config['training']['epochs']}")


---
## 5️⃣ 프로젝트 계획서 작성

지금까지 정한 내용을 종합하여 **프로젝트 계획서**를 작성합니다.

> 💡 **Hint**: 아래 딕셔너리의 빈 항목을 모두 채워주세요. 이 계획서는 이후 모든 노트북에서 참조됩니다.

In [ ]:
# =====================================================
# 📋 프로젝트 계획 종합 및 저장
# =====================================================
import json, os

project_plan = {
    "project_name": "AI/ML 기술 Q&A sLLM",
    "domain": "AI/ML 기술",
    "task": "AI/ML 관련 기술 질문에 정확하게 답변하는 모델",
    
    "model_config": {
        "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
        "method": "LoRA",
        "quantization": "none (FP16)",  # RTX 4060: NF4 호환 문제로 양자화 미사용
        "torch_dtype": "float16",
        "lora_r": 16,
        "lora_alpha": 32,
    },
    
    "data_config": {
        "num_samples": 50,          # 생성할 데이터 수
        "system_prompt": "당신은 AI/ML 분야의 전문가입니다. 정확하고 이해하기 쉽게 답변해주세요.",
        "seed_topics": [
            "LoRA 파인튜닝의 원리와 장점",
            "GPU 메모리 최적화 방법",
            "트랜스포머의 Attention 메커니즘",
            "Python 프로그래밍 기초",
            "딥러닝 학습 기법 (옵티마이저, 스케줄러)",
            "LLM 환각 문제와 해결 방법",
            "RAG와 벡터 데이터베이스",
            "모델 양자화와 서빙",
            "데이터 전처리와 증강",
            "강화학습(DPO, GRPO) 개념",
        ],
    },
    
    "training_config": {
        "num_epochs": 15,
        "batch_size": 1,
        "gradient_accumulation": 8,
        "learning_rate": 2e-4,
        "max_length": 1024,
    },
    
    "eval_prompts": [
        "LoRA 파인튜닝이 무엇인지 설명하세요.",
        "GPU 메모리가 부족할 때 해결 방법을 알려주세요.",
        "트랜스포머의 Attention 메커니즘을 쉽게 설명하세요.",
        "Python에서 리스트와 튜플의 차이점은?",
        "다음 문장을 영어로 번역하세요: 오늘 날씨가 좋습니다.",
    ],
}

# 저장
OUTPUT_DIR = "./output/project"
os.makedirs(OUTPUT_DIR, exist_ok=True)
plan_path = os.path.join(OUTPUT_DIR, "project_plan.json")

with open(plan_path, "w", encoding="utf-8") as f:
    json.dump(project_plan, f, ensure_ascii=False, indent=2)

print("✅ 프로젝트 계획 저장 완료!")
print(f"📌 저장 위치: {plan_path}")
print(f"\n📋 프로젝트 요약:")
print(f"   도메인: {project_plan['domain']}")
print(f"   모델: {project_plan['model_config']['base_model']} (FP16, 양자화 미사용)")
print(f"   생성할 데이터: {project_plan['data_config']['num_samples']}건")
print(f"   에포크: {project_plan['training_config']['num_epochs']}")
print(f"\n💡 수강생은 domain, seed_topics, eval_prompts를 자신의 프로젝트에 맞게 수정하세요!")


In [ ]:
# 저장된 계획 확인
with open("./output/project/project_plan.json", "r", encoding="utf-8") as f:
    saved_plan = json.load(f)

print("📋 저장된 프로젝트 계획:")
print(json.dumps(saved_plan, ensure_ascii=False, indent=2))


---
## 6️⃣ 타임라인 및 체크리스트

### 프로젝트 타임라인

| 일차 | 단계 | 주요 활동 | 산출물 |
|------|------|----------|--------|
| **Day 1** | 기획 | 도메인 선택, 문제 정의, 데이터 전략 | 프로젝트 계획서 |
| **Day 2** | 데이터 | 데이터 수집/정제/증강 파이프라인 | 학습용 데이터셋 |
| **Day 3-4** | 학습 | LoRA/QLoRA 학습, 하이퍼파라미터 튜닝 | 파인튜닝된 모델 |
| **Day 4** | 평가 | 자동/수동 평가, 개선 전략 수립 | 평가 보고서 |
| **Day 5-6** | 배포 | 양자화, API 서버, Streamlit 앱 | 배포된 어플리케이션 |
| **Day 6** | 발표 | 최종 발표, 프로젝트 회고 | 발표 자료 |

In [ ]:
# =====================================================
# ✅ 프로젝트 체크리스트
# =====================================================
checklist = {
    "도메인 선택": True,
    "문제 정의": True,
    "데이터 전략 수립": True,
    "모델 선택": True,
    "평가 기준 정의": True,
    "project_plan.json 저장": True,
}

print("✅ 프로젝트 기획 체크리스트")
print("="*40)
for item, done in checklist.items():
    status = "✅" if done else "❌"
    print(f"  {status} {item}")
print(f"\n🎉 기획 완료! 다음 단계: 33번 노트북 (데이터 파이프라인)")


---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 프로젝트 도메인 선택
- ✅ 문제 정의 (문제, 목표, 평가 기준)
- ✅ 데이터 전략 수립 (수집 방법, 양, 형식)
- ✅ 모델 및 학습 방법론 선택
- ✅ 프로젝트 계획서 작성 및 저장

### 다음 세션 준비사항
- 📌 `project_plan.json` 파일이 저장되어 있는지 확인
- 📌 데이터 소스 URL/경로를 미리 조사해 두기
- 📌 HuggingFace에서 관련 데이터셋 검색해 두기

### 다음 노트북
👉 **33_project_data_pipeline.ipynb**: 프로젝트 데이터 파이프라인 구축